## MERSCOPE Cellpose-SAM tuning on a small ROI

This notebook mirrors the early steps of `MERSCOPE_Resegmentation_Tuning_ROI.ipynb` up through **ROI crop construction**, then runs a **parameter sweep for Cellpose-SAM** and plots a grid of resulting masks.

What you can do here:
- Pick an ROI in **global microns**.
- Build a z-range max projection and crop to ROI.
- Run **Cellpose-SAM** (`pretrained_model='cpsam'`) over a **grid** of key parameters.
- Save per-run outputs and plot comparisons in a grid.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from typing import Tuple

import json
import re
import itertools
import math

import numpy as np
import pandas as pd
import dask
import dask.array as da
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.transform import resize
from skimage.segmentation import find_boundaries
from skimage.measure import label, regionprops

import spatialdata as sd
from cellpose import models

sns.set_context("notebook")
plt.rcParams["figure.figsize"] = (8, 5)

# Recommended for some spatialdata+dask stacks.
dask.config.set({"dataframe.query-planning": False})

print("cellpose CellposeModel.eval signature:")
import inspect
print(inspect.signature(models.CellposeModel().eval))


In [ ]:
# ---- Configure paths ----
ZARR_PATH = Path('/media/mathieubo/SSD2/MerXen/P7513/MOSAIK_analysis/region_R2_mosaik_patched.zarr')
MANIFEST_PATH = ZARR_PATH / 'manifest.json'
TRANSFORM_PATH = ZARR_PATH / 'micron_to_mosaic_pixel_transform.csv'

print('ZARR_PATH exists:', ZARR_PATH.exists())
print('TRANSFORM_PATH exists:', TRANSFORM_PATH.exists())
print('MANIFEST_PATH exists:', MANIFEST_PATH.exists())


In [ ]:
# Load SpatialData and basic metadata.
sdata = sd.read_zarr(ZARR_PATH)
manifest = json.loads(MANIFEST_PATH.read_text())
microns_per_pixel = float(manifest['microns_per_pixel'])
M = np.loadtxt(TRANSFORM_PATH)  # micron_to_mosaic_pixel_transform
Minv = np.linalg.inv(M)

print('Images:', list(sdata.images.keys())[:5], '...')
print('Points:', list(sdata.points.keys()))
print('Shapes:', list(sdata.shapes.keys()))
print('Tables:', list(sdata.tables.keys()))
print('microns_per_pixel:', microns_per_pixel)
print('transform matrix:\n', M)


In [ ]:
def list_plane_keys(images, prefix=None):
    pat = re.compile(r"^(?P<prefix>.+)_z(?P<z>\d+)$")
    out = []
    for k in images.keys():
        m = pat.match(str(k))
        if not m:
            continue
        if prefix is not None and not str(k).startswith(prefix):
            continue
        out.append((int(m.group("z")), str(k)))
    return sorted(out)


def global_to_pixel(x_global, y_global, M):
    arr = np.vstack([x_global, y_global, np.ones_like(x_global)])
    out = M @ arr
    return out[0], out[1]


def make_img8_from_crop(crop: np.ndarray) -> np.ndarray:
    """Convert ROI crop (y,x,c) float/int to 3-ch uint8 for Cellpose."""
    img = crop.astype(np.float32)
    p2, p98 = np.percentile(img, (2, 98))
    img = np.clip((img - p2) / (p98 - p2 + 1e-8), 0, 1)
    img8 = (img * 255).astype(np.uint8)

    if img8.ndim == 2:
        img8 = np.stack([img8] * 3, axis=-1)
    elif img8.shape[-1] == 1:
        img8 = np.repeat(img8, 3, axis=-1)
    elif img8.shape[-1] == 2:
        img8 = np.concatenate([img8, np.zeros_like(img8[..., :1])], axis=-1)
    elif img8.shape[-1] > 3:
        img8 = img8[..., :3]

    return img8


def overlay_boundaries(gray_img: np.ndarray, boundary_mask: np.ndarray, rgb=(255, 0, 0)) -> np.ndarray:
    """Create an RGB overlay of boundaries on a grayscale image."""
    base = gray_img.copy()
    if base.ndim == 3:
        base = base[..., 0]
    base_rgb = np.stack([base, base, base], axis=-1)
    base_rgb[boundary_mask.astype(bool)] = np.array(rgb, dtype=np.uint8)
    return base_rgb


def ensure_mask_in_roi_pixels(masks: np.ndarray, scale_factor: float, target_hw: Tuple[int, int]) -> np.ndarray:
    """If Cellpose ran on a downscaled image, upsample masks back to ROI pixels."""
    if scale_factor is None or float(scale_factor) <= 1.0:
        return masks

    th, tw = target_hw
    # nearest-neighbor upsample for labels
    up = resize(
        masks.astype(np.int32),
        (th, tw),
        order=0,
        preserve_range=True,
        anti_aliasing=False,
    ).astype(np.int32)
    return up


In [ ]:
# ---- Dataset-level overview for ROI selection ----
points_key = list(sdata.points.keys())[0]
pts_dd = sdata.points[points_key]

sample_frac = 0.005  # increase for denser overview
overview = pts_dd.sample(frac=sample_frac, random_state=42)[['x', 'y', 'z', 'gene']].compute()

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(overview['x'], overview['y'], s=0.2, alpha=0.3)
ax.set_title(f'Global transcript overview (sample frac={sample_frac})')
ax.set_xlabel('global x (microns)')
ax.set_ylabel('global y (microns)')
ax.set_aspect('equal')
plt.show()


In [ ]:
# ---- Set your ROI in GLOBAL micron coordinates ----
# Edit these values iteratively and rerun downstream cells.
ROI = {
    'x_min': 5000.0,
    'x_max': 5400.0,
    'y_min': 3500.0,
    'y_max': 3900.0,
}

z_start = 0
z_end = 6
channel_names_to_use = ['DAPI', 'PolyT']  # subset from available channels

print(ROI)
print('z range:', z_start, z_end)
print('channels:', channel_names_to_use)


In [ ]:
# Visualize selected ROI over transcript overview.
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(overview['x'], overview['y'], s=0.2, alpha=0.25)
rect_x = [ROI['x_min'], ROI['x_max'], ROI['x_max'], ROI['x_min'], ROI['x_min']]
rect_y = [ROI['y_min'], ROI['y_min'], ROI['y_max'], ROI['y_max'], ROI['y_min']]
ax.plot(rect_x, rect_y, color='red', linewidth=2)
ax.set_title('ROI on global transcript overview')
ax.set_xlabel('global x (microns)')
ax.set_ylabel('global y (microns)')
ax.set_aspect('equal')
plt.show()


In [ ]:
# ---- Build z-range max projection and crop to ROI (lazy ROI reads) ----
plane_keys = list_plane_keys(sdata.images)

# Try to auto-detect the image prefix from keys like "..._z0".
img_prefix = None
if plane_keys:
    img_prefix = re.sub(r"_z\d+$", "", plane_keys[0][1])

selected_keys = [k for z, k in plane_keys if z_start <= z <= z_end and (img_prefix is None or k.startswith(img_prefix))]
print('selected plane keys:', selected_keys)

if not selected_keys:
    raise ValueError('No image planes selected for the z range.')

# Convert ROI global microns -> mosaic pixel bbox first.
xpix, ypix = global_to_pixel(
    np.array([ROI['x_min'], ROI['x_max']]),
    np.array([ROI['y_min'], ROI['y_max']]),
    M,
)
x0, x1 = int(np.floor(min(xpix))), int(np.ceil(max(xpix)))
y0, y1 = int(np.floor(min(ypix))), int(np.ceil(max(ypix)))

# Clamp using first selected plane shape without loading pixel data.
img0 = sdata.images[selected_keys[0]]
img0_xr = img0['scale0'].ds['image'] if hasattr(img0, '__contains__') and 'scale0' in img0 else img0
if all(d in img0_xr.dims for d in ('y', 'x')):
    h = int(img0_xr.sizes['y'])
    w = int(img0_xr.sizes['x'])
else:
    raise ValueError(f'Unexpected image dims for ROI crop: {img0_xr.dims}')

x0 = max(0, x0)
y0 = max(0, y0)
x1 = min(w, x1)
y1 = min(h, y1)

print('pixel bbox:', (x0, x1, y0, y1), 'size=', (x1 - x0, y1 - y0))

if x1 <= x0 or y1 <= y0:
    raise ValueError('ROI maps to an empty pixel crop. Adjust ROI or transform.')

lazy_roi_planes = []
channels = None
use_ch = None

for key in selected_keys:
    img_obj = sdata.images[key]
    img_xr = img_obj['scale0'].ds['image'] if hasattr(img_obj, '__contains__') and 'scale0' in img_obj else img_obj

    if not all(d in img_xr.dims for d in ('y', 'x', 'c')):
        raise ValueError(f'Expected image dims to include c,y,x. Got {img_xr.dims} for {key}')

    c_all = [str(c) for c in img_xr.coords['c'].values] if 'c' in img_xr.coords else None
    if channels is None:
        channels = c_all

    roi_xr = img_xr.isel(y=slice(y0, y1), x=slice(x0, x1))

    if channel_names_to_use is not None and c_all is not None:
        keep = [c for c in channel_names_to_use if c in c_all]
        if not keep:
            raise ValueError('No selected channels found in image channel coords.')
        roi_xr = roi_xr.sel(c=keep)
        if use_ch is None:
            use_ch = keep
    else:
        if use_ch is None:
            use_ch = c_all

    lazy_roi_planes.append(roi_xr.transpose('y', 'x', 'c').data)

proj = da.max(da.stack(lazy_roi_planes, axis=0), axis=0).compute()  # (y, x, c)
crop = proj

print('projection/crop shape:', crop.shape)
print('available channels:', channels)
print('channels used:', use_ch)

fig, ax = plt.subplots(figsize=(7, 7))
show = crop[..., 0] if crop.shape[-1] > 0 else crop.squeeze()
ax.imshow(show, cmap='gray')
ax.set_title('ROI crop preview (first channel)')
ax.axis('off')
plt.show()


### What to sweep for Cellpose-SAM

From the Cellpose documentation, the parameters that most strongly change masks are typically:
- **`cellprob_threshold`**: how strict the model is about calling pixels "cell".
- **`flow_threshold`**: how strict the model is about accepting a mask given flow consistency.
- **`diameter` / `rescale`**: effective object size prior / scaling (useful when cells are much larger/smaller than typical).
- **`min_size`**: post-filter for small objects (removes tiny false positives).

Other parameters matter more for speed / tiling than mask shape (e.g. `bsize`, `tile_overlap`) unless you see tile-edge artifacts.


In [ ]:
# ---- Prepare Cellpose-SAM input image ----
img8 = make_img8_from_crop(crop)
print('img8 shape:', img8.shape, img8.dtype)

# Optional: run Cellpose on a downscaled image for speed.
# If you set factor_rescale>1, masks will be upsampled back to ROI pixels for plotting.
factor_rescale = 1.0

if factor_rescale > 1.0:
    target_shape = (
        int(img8.shape[0] / factor_rescale),
        int(img8.shape[1] / factor_rescale),
        img8.shape[2],
    )
    img_seg = resize(img8, target_shape, preserve_range=True, anti_aliasing=True).astype(np.uint8)
else:
    img_seg = img8

print('img_seg shape:', img_seg.shape, img_seg.dtype)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(img_seg[..., 0], cmap='gray')
ax.set_title('Cellpose-SAM input (channel 0)')
ax.axis('off')
plt.show()


In [ ]:
# ---- Define Cellpose-SAM sweep ----
# Keep the default sweep small to start; expand once you see the right direction.

cellpose_param_sets_prev = []  # keep old sweeps here if desired

# Primary sweep: (cellprob_threshold x flow_threshold)
cellpose_param_sets = []
cellprob_values = [-6.0, -4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0]
flow_values = [0.2, 0.4, 0.6, 0.8]

# Fixed parameters (edit as needed)
base_params = {
    'gpu': True,
    'pretrained_model': 'cpsam',
    'diameter': None,
    'min_size': 15,
    'augment': False,
    'normalize': True,
    'invert': False,
    'tile_overlap': 0.10,
    'bsize': 256,
}

for cp in cellprob_values:
    for ft in flow_values:
        cellpose_param_sets.append({
            **base_params,
            'cellprob_threshold': float(cp),
            'flow_threshold': float(ft),
            'label': f'cp{cp:g}__ft{ft:g}',
        })

print('Cellpose-SAM runs:', len(cellpose_param_sets))
print('Example params:', cellpose_param_sets[0])


In [ ]:
# ---- Run Cellpose-SAM sweep ----
# Saves per-cell label masks (masks.npz) and per-cell boundary masks (boundaries.npz).

run_sweep = True
out_dir = Path('./tmp_cellpose_sam_sweep')
out_dir.mkdir(parents=True, exist_ok=True)

# Construct model once (fastest), then sweep eval-time parameters.
# Note: in Cellpose v4+, CellposeModel defaults to pretrained_model='cpsam'.
model = models.CellposeModel(
    gpu=bool(base_params['gpu']),
    pretrained_model=str(base_params.get('pretrained_model', 'cpsam')),
)

cellpose_results = []

for i, params in enumerate(cellpose_param_sets, 1):
    label = params['label']
    run_dir = out_dir / label
    run_dir.mkdir(parents=True, exist_ok=True)
    masks_path = run_dir / 'masks.npz'
    boundary_path = run_dir / 'boundaries.npz'
    meta_path = run_dir / 'params.json'

    if masks_path.exists() and meta_path.exists():
        # fast resume
        try:
            with open(meta_path, 'r') as f:
                params_out = json.load(f)
        except Exception:
            params_out = dict(params)

        cellpose_results.append({
            'label': label,
            'params': params_out,
            'masks_path': masks_path,
            'boundary_path': boundary_path,
            'scale_factor': float(factor_rescale),
            'n_masks': int(params_out.get('n_masks')) if 'n_masks' in params_out else None,
        })
        continue

    if not run_sweep:
        continue

    masks, flows, styles = model.eval(
        img_seg,
        diameter=params.get('diameter'),
        flow_threshold=float(params['flow_threshold']),
        cellprob_threshold=float(params['cellprob_threshold']),
        min_size=int(params.get('min_size', 15)),
        stitch_threshold=float(params.get('stitch_threshold', 0.0)),
        augment=bool(params.get('augment', False)),
        normalize=bool(params.get('normalize', True)),
        invert=bool(params.get('invert', False)),
        tile_overlap=float(params.get('tile_overlap', 0.10)),
        bsize=int(params.get('bsize', 256)),
        progress=None,
    )

    # Map masks back to ROI pixel coords if we downscaled.
    masks_roi = ensure_mask_in_roi_pixels(masks, scale_factor=factor_rescale, target_hw=img8.shape[:2])

    n_masks = int(masks_roi.max())

    # Save per-cell integer label array so boundaries can be recomputed per-cell.
    np.savez_compressed(masks_path, masks=masks_roi.astype(np.int32))

    # find_boundaries on the labeled (not binarised) array marks edges around
    # each individual cell, including boundaries between adjacent cells.
    boundary = find_boundaries(masks_roi, mode='outer')
    np.savez_compressed(boundary_path, boundary=boundary.astype(np.uint8))

    params_out = dict(params)
    params_out['n_masks'] = n_masks
    params_out['mask_shape'] = [int(masks_roi.shape[0]), int(masks_roi.shape[1])]

    with open(meta_path, 'w') as f:
        json.dump(params_out, f, indent=2)

    cellpose_results.append({
        'label': label,
        'params': params_out,
        'masks_path': masks_path,
        'boundary_path': boundary_path,
        'scale_factor': float(factor_rescale),
        'n_masks': n_masks,
    })

    print(f'[{i}/{len(cellpose_param_sets)}] done:', label, '| n_masks:', n_masks)

print('Available results:', len(cellpose_results))


In [ ]:
# ---- Plot a grid of Cellpose-SAM results ----
# This plots the (cellprob_threshold x flow_threshold) sweep as a grid.

if 'cellpose_results' not in globals() or len(cellpose_results) == 0:
    raise ValueError('No Cellpose sweep results found. Run the sweep cell first.')

# Load per-cell boundary masks.
# Prefer masks.npz (per-cell labels -> accurate per-cell boundaries) over
# the old binary boundaries.npz (outer cluster edge only).
label_to_boundary = {}
for res in cellpose_results:
    masks_path = res.get('masks_path', res['boundary_path'].parent / 'masks.npz')
    if Path(masks_path).exists():
        arr = np.load(masks_path)
        masks = arr['masks'].astype(np.int32)
        # Boundaries computed from the labeled array mark each individual cell's
        # edge, including the boundary between two adjacent cells.
        label_to_boundary[res['label']] = find_boundaries(masks, mode='outer')
    elif Path(res['boundary_path']).exists():
        arr = np.load(res['boundary_path'])
        label_to_boundary[res['label']] = arr['boundary'].astype(bool)

label_to_params = {r['label']: r['params'] for r in cellpose_results}

cp_vals = sorted({float(label_to_params[l]['cellprob_threshold']) for l in label_to_params})
ft_vals = sorted({float(label_to_params[l]['flow_threshold']) for l in label_to_params})

ncols = len(ft_vals)
nrows = len(cp_vals)
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), constrained_layout=True)
if nrows == 1 and ncols == 1:
    axes = np.array([[axes]])
elif nrows == 1:
    axes = axes[np.newaxis, :]
elif ncols == 1:
    axes = axes[:, np.newaxis]

for ri, cp in enumerate(cp_vals):
    for ci, ft in enumerate(ft_vals):
        ax = axes[ri, ci]
        lbl = f'cp{cp:g}__ft{ft:g}'
        boundary = label_to_boundary.get(lbl)
        n_masks = label_to_params[lbl].get('n_masks', '?') if lbl in label_to_params else '?'

        ax.imshow(img8[..., 0], cmap='gray_r')
        if boundary is not None and boundary.any():
            from skimage.morphology import binary_dilation, disk
            thick = binary_dilation(boundary, footprint=disk(3))
            overlay = np.zeros((*thick.shape, 4), dtype=np.float32)
            overlay[thick, 1] = 1.0   # green channel
            overlay[thick, 3] = 1.0   # fully opaque
            ax.imshow(overlay)

        ax.set_title(f'cp={cp:g}, ft={ft:g}\nn={n_masks}', fontsize=8)
        ax.set_aspect('equal')
        ax.axis('off')

plt.show()
plt.savefig('cellpose_sam_sweep.png', dpi=100)

## Fine sweep — focused on the best region

Based on the coarse grid, `cellprob_threshold` ≈ −1 to −3 with `flow_threshold` ≥ 0.8 gives the most cells.  
This fine sweep narrows the `cellprob` range and pushes `flow_threshold` higher to find the plateau.

In [ ]:
# ---- Define fine Cellpose-SAM sweep ----
# Narrow cellprob range; extend flow_threshold above 0.8 to find the plateau.

fine_cellprob_values = [-6.0, -5.0, -4.0, -3.0, -2.5, -2.0, -1.5, -1.0]
fine_flow_values     = [0.8, 1.0, 1.2, 1.4, 1.6]

fine_cellpose_param_sets = []
for cp in fine_cellprob_values:
    for ft in fine_flow_values:
        fine_cellpose_param_sets.append({
            **base_params,
            'cellprob_threshold': float(cp),
            'flow_threshold':     float(ft),
            'label': f'cp{cp:g}__ft{ft:g}',
        })

# Construct model once (fastest), then sweep eval-time parameters.
# Note: in Cellpose v4+, CellposeModel defaults to pretrained_model='cpsam'.
model = models.CellposeModel(
    gpu=bool(base_params['gpu']),
    pretrained_model=str(base_params.get('pretrained_model', 'cpsam')),
)

print('Fine sweep runs:', len(fine_cellpose_param_sets))

In [ ]:
# ---- Run fine Cellpose-SAM sweep ----

from tqdm.notebook import tqdm
import torch

# Guard: ensure model is on GPU. Re-create if missing or on CPU.
if 'model' not in dir() or not hasattr(model, 'device') or str(getattr(model, 'device', 'cpu')) == 'cpu':
    if not torch.cuda.is_available():
        print('WARNING: CUDA not available — running on CPU (will be slow).')
    model = models.CellposeModel(
        gpu=torch.cuda.is_available(),
        pretrained_model=str(base_params.get('pretrained_model', 'cpsam')),
    )
print(f'Model device: {model.device}')

run_fine_sweep = True
fine_out_dir = Path('./tmp_cellpose_sam_sweep_fine')
fine_out_dir.mkdir(parents=True, exist_ok=True)

fine_cellpose_results = []

pbar = tqdm(list(enumerate(fine_cellpose_param_sets, 1)), desc='Fine sweep', unit='run')
for i, params in pbar:
    label = params['label']
    run_dir = fine_out_dir / label
    run_dir.mkdir(parents=True, exist_ok=True)
    masks_path    = run_dir / 'masks.npz'
    boundary_path = run_dir / 'boundaries.npz'
    meta_path     = run_dir / 'params.json'

    if masks_path.exists() and meta_path.exists():
        try:
            with open(meta_path, 'r') as f:
                params_out = json.load(f)
        except Exception:
            params_out = dict(params)

        fine_cellpose_results.append({
            'label':        label,
            'params':       params_out,
            'masks_path':   masks_path,
            'boundary_path': boundary_path,
            'scale_factor': float(factor_rescale),
            'n_masks':      int(params_out.get('n_masks')) if 'n_masks' in params_out else None,
        })
        pbar.set_postfix(label=label, n=params_out.get('n_masks', '?'), cached=True)
        continue

    if not run_fine_sweep:
        continue

    pbar.set_postfix(label=label, status='running...')

    masks, flows, styles = model.eval(
        img_seg,
        diameter=params.get('diameter'),
        flow_threshold=float(params['flow_threshold']),
        cellprob_threshold=float(params['cellprob_threshold']),
        min_size=int(params.get('min_size', 15)),
        stitch_threshold=float(params.get('stitch_threshold', 0.0)),
        augment=bool(params.get('augment', False)),
        normalize=bool(params.get('normalize', True)),
        invert=bool(params.get('invert', False)),
        tile_overlap=float(params.get('tile_overlap', 0.10)),
        bsize=int(params.get('bsize', 256)),
        progress=None,
    )

    masks_roi = ensure_mask_in_roi_pixels(masks, scale_factor=factor_rescale, target_hw=img8.shape[:2])
    n_masks   = int(masks_roi.max())

    np.savez_compressed(masks_path, masks=masks_roi.astype(np.int32))

    boundary = find_boundaries(masks_roi, mode='outer')
    np.savez_compressed(boundary_path, boundary=boundary.astype(np.uint8))

    params_out = dict(params)
    params_out['n_masks']    = n_masks
    params_out['mask_shape'] = [int(masks_roi.shape[0]), int(masks_roi.shape[1])]

    with open(meta_path, 'w') as f:
        json.dump(params_out, f, indent=2)

    fine_cellpose_results.append({
        'label':        label,
        'params':       params_out,
        'masks_path':   masks_path,
        'boundary_path': boundary_path,
        'scale_factor': float(factor_rescale),
        'n_masks':      n_masks,
    })

    pbar.set_postfix(label=label, n=n_masks)

print(f'Fine sweep complete: {len(fine_cellpose_results)} results available.')

In [ ]:
# ---- Plot fine sweep grid ----

if 'fine_cellpose_results' not in globals() or len(fine_cellpose_results) == 0:
    raise ValueError('No fine sweep results found. Run the fine sweep cell first.')

from skimage.morphology import binary_dilation, disk

label_to_boundary_fine = {}
for res in fine_cellpose_results:
    masks_path = res.get('masks_path', res['boundary_path'].parent / 'masks.npz')
    if Path(masks_path).exists():
        arr = np.load(masks_path)
        masks = arr['masks'].astype(np.int32)
        label_to_boundary_fine[res['label']] = find_boundaries(masks, mode='outer')
    elif Path(res['boundary_path']).exists():
        arr = np.load(res['boundary_path'])
        label_to_boundary_fine[res['label']] = arr['boundary'].astype(bool)

label_to_params_fine = {r['label']: r['params'] for r in fine_cellpose_results}

cp_vals = sorted({float(label_to_params_fine[l]['cellprob_threshold']) for l in label_to_params_fine})
ft_vals = sorted({float(label_to_params_fine[l]['flow_threshold'])     for l in label_to_params_fine})

ncols = len(ft_vals)
nrows = len(cp_vals)
fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 4*nrows), constrained_layout=True)
if nrows == 1 and ncols == 1:
    axes = np.array([[axes]])
elif nrows == 1:
    axes = axes[np.newaxis, :]
elif ncols == 1:
    axes = axes[:, np.newaxis]

for ri, cp in enumerate(cp_vals):
    for ci, ft in enumerate(ft_vals):
        ax  = axes[ri, ci]
        lbl = f'cp{cp:g}__ft{ft:g}'
        boundary = label_to_boundary_fine.get(lbl)
        n_masks  = label_to_params_fine[lbl].get('n_masks', '?') if lbl in label_to_params_fine else '?'

        ax.imshow(img8[..., 0], cmap='gray_r')
        if boundary is not None and boundary.any():
            thick = binary_dilation(boundary, footprint=disk(3))
            overlay = np.zeros((*thick.shape, 4), dtype=np.float32)
            overlay[thick, 1] = 1.0
            overlay[thick, 3] = 1.0
            ax.imshow(overlay)

        ax.set_title(f'cp={cp:g}, ft={ft:g}\nn={n_masks}', fontsize=8)
        ax.set_aspect('equal')
        ax.axis('off')

plt.show()

In [ ]:
# ---- Plot single fine-sweep result (zoomable) ----

if 'fine_cellpose_results' not in globals() or len(fine_cellpose_results) == 0:
    raise ValueError('No fine sweep results found. Run the fine sweep cell first.')

from pathlib import Path
from skimage.segmentation import find_boundaries
from skimage.morphology import binary_dilation, disk

target_cp = -1.0
target_ft = 0.8
label = f'cp{target_cp:g}__ft{target_ft:g}'

default_xlim = (0, img8.shape[1] - 1)
default_ylim = (img8.shape[0] - 1, 0)
xlim = default_xlim
ylim = default_ylim

if 'label_to_boundary_fine' not in globals():
    label_to_boundary_fine = {}

# If this specific label is missing, load it directly from fine_cellpose_results
if label not in label_to_boundary_fine:
    match = next((r for r in fine_cellpose_results if r.get('label') == label), None)
    if match is None:
        raise ValueError(f'Label {label} not found in fine_cellpose_results.')

    masks_path = Path(match.get('masks_path', match['boundary_path'].parent / 'masks.npz'))
    boundary_path = Path(match['boundary_path'])

    if masks_path.exists():
        arr = np.load(masks_path)
        masks = arr['masks'].astype(np.int32)
        label_to_boundary_fine[label] = find_boundaries(masks, mode='outer')
    elif boundary_path.exists():
        arr = np.load(boundary_path)
        # support either key just in case
        if 'boundary' in arr:
            label_to_boundary_fine[label] = arr['boundary'].astype(bool)
        elif 'boundaries' in arr:
            label_to_boundary_fine[label] = arr['boundaries'].astype(bool)
        else:
            raise ValueError(f'No boundary array found in {boundary_path}. Keys: {list(arr.keys())}')
    else:
        raise ValueError(f'Neither masks nor boundary file exists for {label}.')

boundary = label_to_boundary_fine.get(label)
if boundary is None:
    raise ValueError(f'Boundary still missing for {label}.')

fig, ax = plt.subplots(figsize=(8, 8), constrained_layout=True)
ax.imshow(img8[..., 0], cmap='gray_r')

thickness = 3

if boundary.any():
    thick = binary_dilation(boundary, footprint=disk(thickness))
    overlay = np.zeros((*thick.shape, 4), dtype=np.float32)
    overlay[thick, 1] = 1.0
    overlay[thick, 3] = 1.0
    ax.imshow(overlay)

ax.set_title(f'Fine sweep: cp={target_cp:g}, ft={target_ft:g}', fontsize=11)
ax.set_aspect('equal')

#xlim = (2500, 3000)
#ylim = (1500, 1000)

ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
ax.tick_params(axis='both', which='both', labelsize=8)
ax.minorticks_on()
ax.grid(True, which='major', color='black', alpha=0.35, linewidth=0.8)
ax.grid(True, which='minor', color='black', alpha=0.15, linewidth=0.5)

plt.show()


In [ ]:
# ---- Plot single fine-sweep result (zoomable) ----

if 'fine_cellpose_results' not in globals() or len(fine_cellpose_results) == 0:
    raise ValueError('No fine sweep results found. Run the fine sweep cell first.')

from skimage.morphology import binary_dilation, disk

# Select the fine-sweep parameter combo to display
target_cp = -6.0
target_ft = 1.4
label = f'cp{target_cp:g}__ft{target_ft:g}'

# Optional zoom window (set to None to show full image)
default_xlim = (0, img8.shape[1] - 1)
default_ylim = (img8.shape[0] - 1, 0)  # inverted so origin stays at top-left
xlim = default_xlim
ylim = default_ylim

# Build boundary lookup if needed
if 'label_to_boundary_fine' not in globals() or not label_to_boundary_fine:
    label_to_boundary_fine = {}
    for res in fine_cellpose_results:
        masks_path = res.get('masks_path', res['boundary_path'].parent / 'masks.npz')
        if Path(masks_path).exists():
            arr = np.load(masks_path)
            masks = arr['masks'].astype(np.int32)
            label_to_boundary_fine[res['label']] = find_boundaries(masks, mode='outer')
        elif Path(res['boundary_path']).exists():
            arr = np.load(res['boundary_path'])
            label_to_boundary_fine[res['label']] = arr['boundary'].astype(bool)

boundary = label_to_boundary_fine.get(label)
if boundary is None:
    raise ValueError(f'No boundary found for {label}. Check target_cp/target_ft and available results.')

fig, ax = plt.subplots(figsize=(8, 8), constrained_layout=True)
ax.imshow(img8[..., 0], cmap='gray_r')

if boundary.any():
    thick = binary_dilation(boundary, footprint=disk(3))
    overlay = np.zeros((*thick.shape, 4), dtype=np.float32)
    overlay[thick, 1] = 1.0
    overlay[thick, 3] = 1.0
    ax.imshow(overlay)

ax.set_title(f'Fine sweep: cp={target_cp:g}, ft={target_ft:g}', fontsize=11)
ax.set_aspect('equal')

# Axes, ticks, and grid for coordinate-aware inspection
ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
ax.tick_params(axis='both', which='both', labelsize=8)
ax.minorticks_on()
ax.grid(True, which='major', color='yellow', alpha=0.35, linewidth=0.8)
ax.grid(True, which='minor', color='yellow', alpha=0.15, linewidth=0.5)

plt.show()


### How `flow_threshold` works — and where to stop raising it

**The check Cellpose runs per candidate mask:**

1. The neural net predicts a 2-channel flow field `dP_net` (∂Y, ∂X) for every pixel — unit-norm vectors pointing toward the cell's centre.
2. An Euler-integration dynamics step pushes pixels along these vectors; pixels that converge to the same point form a candidate mask.
3. For each candidate mask, Cellpose *recomputes* the ideal flow `dP_mask` from scratch via heat-diffusion from that mask's centre.
4. It then measures the **mean squared error (MSE)** between those two flows, summed over Y and X:

$$\text{flow\_error} = \sum_{\text{axis}} \overline{\left(\hat{f}^{\text{mask}} - \hat{f}^{\text{net}}/5\right)^2}$$

   Masks whose error exceeds `flow_threshold` are **discarded**.

**What limits the scale?**

Both flow vectors are unit-norm. For a single axis, the worst-case squared difference between two opposing unit vectors is $(+1 - (-1))^2 = 4$. Summed over Y and X, the **theoretical maximum is ≈ 4**.

In practice:
- **< 0.4** (default): only geometrically consistent, well-shaped masks survive.
- **0.4 – 1.0**: increasingly distorted/elongated cells are accepted; usually still real biology.
- **1.0 – 1.5**: mostly stray convergence artefacts; genuine new cells become rare.
- **> 1.5**: the filter is essentially off — you're accepting random pixel clusters.

The cell below plots **n_masks vs flow_threshold** for each `cellprob` value to show empirically where your data's curve flattens.

In [ ]:
# ---- n_masks vs flow_threshold: find the diminishing-returns point ----
# Pools both the coarse and fine sweeps so the full ft range [0.2 → 1.6] is visible.

all_results = []
for res in globals().get('cellpose_results', []):
    all_results.append({
        'cp':  float(res['params']['cellprob_threshold']),
        'ft':  float(res['params']['flow_threshold']),
        'n':   int(res['params'].get('n_masks', 0) or 0),
        'sweep': 'coarse',
    })
for res in globals().get('fine_cellpose_results', []):
    all_results.append({
        'cp':  float(res['params']['cellprob_threshold']),
        'ft':  float(res['params']['flow_threshold']),
        'n':   int(res['params'].get('n_masks', 0) or 0),
        'sweep': 'fine',
    })

if not all_results:
    raise ValueError('Run at least one sweep first.')

df_all = pd.DataFrame(all_results).drop_duplicates(subset=['cp', 'ft']).sort_values(['cp', 'ft'])

cp_plot_vals = sorted(df_all['cp'].unique())
palette = sns.color_palette('tab10', len(cp_plot_vals))

fig, ax = plt.subplots(figsize=(9, 5))

for color, cp in zip(palette, cp_plot_vals):
    sub = df_all[df_all['cp'] == cp].sort_values('ft')
    ax.plot(sub['ft'], sub['n'], marker='o', color=color, label=f'cp={cp:g}', linewidth=1.8)

# Mark the theoretical hard ceiling
ax.axvline(x=4.0, color='gray', linestyle=':', linewidth=1, label='Theoretical max (≈4)')

# Shade the "diminishing returns" zone
ax.axvspan(1.0, 1.5, alpha=0.08, color='orange', label='Diminishing-returns zone')
ax.axvspan(1.5, max(df_all['ft'].max() + 0.2, 2.0), alpha=0.08, color='red', label='Effectively no filter')

ax.set_xlabel('flow_threshold')
ax.set_ylabel('n_masks')
ax.set_title('n_masks vs flow_threshold\n(curve flattening = diminishing returns)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
sns.despine()
plt.tight_layout()
plt.show()